# DataFramer SDK — PII/PHI Anonymization Workflow

This notebook demonstrates how to use the [DataFramer](https://dataframer.ai) Python [SDK](https://pypi.org/project/pydataframer) (PIP package: `pydataframer`) to detect and mask **Personally Identifiable Information (PII)** and **Protected Health Information (PHI)** in your datasets.

We will walk through the complete workflow:
- Upload a seed dataset containing sensitive data
- Create a transform job to detect and mask PII/PHI entities
- Inspect the masked output sample by sample
- Download all anonymized files as a ZIP archive

The anonymization pipeline uses the `AIMon-PII-M1` model combined with heuristic rules (`gliner+heuristics`) for high-accuracy detection across names, emails, phone numbers, SSNs, dates of birth, and more.

## Setup

Install the [DataFramer SDK](https://pypi.org/project/pydataframer) and additional utilities.

In [ ]:
%%capture
%pip install --upgrade pydataframer tenacity pandas requests

A DataFramer API key is required. Retrieve yours from **Account → Keys → Copy API Key** on the [web application](https://app.aimon.ai) and add it as a Colab secret named `DATAFRAMER_API_KEY`.

In [ ]:
import os
from google.colab import userdata

os.environ['DATAFRAMER_API_KEY'] = userdata.get('DATAFRAMER_API_KEY')

In [ ]:
import io
import os
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import dataframer
import pandas as pd
import requests
from dataframer import Dataframer
from tenacity import retry, retry_if_result, stop_never, wait_fixed

BASE_URL = "https://df-api.dataframer.ai"
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"run_id: {run_id}")

## Create Sample Dataset

We build a small synthetic CSV in-memory — no external files required. Each row is a fictitious patient support record containing PII/PHI fields that the transform job will detect and mask.

In [ ]:
CSV_DATA = """\
patient_id,first_name,last_name,email,phone,dob,ssn,notes
P001,John,Smith,john.smith@email.com,555-867-5309,1985-03-14,123-45-6789,Patient John Smith reports mild chest pain. Born 1985-03-14. SSN 123-45-6789.
P002,Maria,Garcia,m.garcia@healthnet.org,555-234-5678,1972-11-28,987-65-4321,Follow-up for Maria Garcia. Contact at m.garcia@healthnet.org or 555-234-5678.
P003,David,Lee,dlee@provider.com,555-321-0987,1990-07-04,456-78-9012,Patient David Lee (DOB 1990-07-04). SSN on file: 456-78-9012.
P004,Sarah,Johnson,sarah.j@clinic.org,555-456-7890,1968-12-25,789-01-2345,Emergency contact for Sarah Johnson: 555-456-7890. Email sarah.j@clinic.org
P005,Robert,Chen,r.chen@medcenter.com,555-654-3210,1955-09-18,321-54-9876,Mr. Robert Chen (DOB 1955-09-18) was seen on 2024-01-15. SSN 321-54-9876.
"""

pd.set_option("display.max_colwidth", 80)
df_preview = pd.read_csv(io.StringIO(CSV_DATA))
print(f"Sample dataset: {len(df_preview)} rows")
df_preview

## DataFramer SDK — PII/PHI Detection & Anonymization

### Step 1: Initialize the Client

Create a DataFramer client using your API key. The key is loaded from the Colab secret set in the **Setup** section.

In [ ]:
client = Dataframer(
    api_key=os.environ["DATAFRAMER_API_KEY"],
    base_url=BASE_URL,
)

print("✅ Dataframer client initialized")
print(f"   SDK version:  {dataframer.__version__}")
print(f"   API key:      {client.api_key[:4]}...")
print(f"   Base URL:     {client.base_url}")

### Step 2: Upload the Dataset

Wrap the synthetic CSV in a ZIP buffer and upload it as a seed dataset. If a dataset with the same name already exists it is reused (idempotent).

In [ ]:
dataset_name = f"anonymize_seed_{run_id}"


def _find_existing_dataset(name):
    all_datasets = client.dataframer.seed_datasets.list()
    return next((d for d in all_datasets if d.name == name), None)


zip_buffer = io.BytesIO()
with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.writestr("patient_records.csv", CSV_DATA)
zip_buffer.seek(0)

try:
    dataset = client.dataframer.seed_datasets.create_from_zip(
        name=dataset_name,
        description="Synthetic patient support records for PII anonymization demo",
        zip_file=zip_buffer,
    )
except Exception as e:
    if "already exists" in str(e):
        dataset = _find_existing_dataset(dataset_name)
        print(f"  ℹ️  Dataset '{dataset_name}' already exists — reusing it")
    else:
        raise

dataset_id = dataset.id

print("✅ Dataset ready")
print(f"   ID:         {dataset_id}")
print(f"   Name:       {dataset.name}")
print(f"   File count: {dataset.file_count}")

### Step 3: Create a Transform Job

Create a transform job that will scan every row in the dataset and replace detected entities with masked tokens.

**Detection method**: `gliner+heuristics` — the recommended setting. Combines the `AIMon-PII-M1` neural model with regex-based heuristic rules for high precision and recall.

**PII types targeted** (see the [full entity catalogue](#appendix-pii-types)):

| Category | Entity types |
|---|---|
| Personal | `first_name`, `last_name`, `date_of_birth` |
| Contact | `email`, `phone_number`, `street_address` |
| Financial | `ssn` |

In [ ]:
job_name = f"anonymize_job_{run_id}"

PII_TYPES = [
    "first_name",
    "last_name",
    "email",
    "phone_number",
    "street_address",
    "date_of_birth",
    "ssn",
]

job = client.dataframer.transform_jobs.create(
    dataset_id=dataset_id,
    name=job_name,
    pii_types=PII_TYPES,
    detection_method="gliner+heuristics",
    threshold=0.3,
    evaluation_model="anthropic/claude-sonnet-4-6",
)

job_id = job.id

print("✅ Transform job created")
print(f"   ID:               {job_id}")
print(f"   Name:             {job.name}")
print(f"   Status:           {job.status}")
print(f"   Detection method: {job.detection_method}")
print(f"   PII types:        {job.pii_types}")

### Step 4: Poll Until Job Completes

The transform job runs asynchronously. We poll every 10 seconds until it reaches `SUCCEEDED` or `FAILED`.

In [ ]:
def job_not_finished(result):
    return result.status not in ("SUCCEEDED", "FAILED")


@retry(wait=wait_fixed(10), retry=retry_if_result(job_not_finished), stop=stop_never)
def poll_job_status(client, job_id):
    result = client.dataframer.transform_jobs.retrieve(job_id)
    print(
        f"[{datetime.now(timezone.utc).isoformat(timespec='seconds')}] "
        f"Job {job_id[:8]}... status: {result.status}",
        flush=True,
    )
    return result


print("Polling for job completion (this may take several minutes)...")
job_result = poll_job_status(client, job_id)

if job_result.status == "FAILED":
    error = (job_result.metrics_json or {}).get("error", "unknown error")
    raise RuntimeError(f"Transform job failed: {error}")

metrics = job_result.metrics_json or {}
samples = metrics.get("transformed_samples", [])

print(f"\n✅ Transform job completed successfully!")
print(f"   Samples transformed:     {len(samples)}")
if samples:
    total_entities = sum(
        sum(v for v in (s.get("entity_summary") or {}).values() if isinstance(v, int))
        for s in samples
    )
    print(f"   Total entities redacted: {total_entities}")

### Step 5: List All Transform Jobs

Retrieve all transform jobs for your company account (newest first). Useful for auditing past anonymization runs.

In [ ]:
print("=" * 80)
print("📋 All Transform Jobs")
print("=" * 80)

all_jobs = client.dataframer.transform_jobs.list()
print(f"Found {len(all_jobs)} transform job(s)\n")

for i, j in enumerate(all_jobs[:5], 1):
    print(f"  Job {i}:")
    print(f"    Name:    {j.name}")
    print(f"    ID:      {j.id}")
    print(f"    Status:  {j.status}")
    print(f"    Created: {j.created_at}")
    print()

if len(all_jobs) > 5:
    print(f"  ... and {len(all_jobs) - 5} more")

### Step 6: Retrieve Full Job Details

Fetch the complete job record including dataset metadata, configuration parameters, and timing information.

In [ ]:
print("=" * 80)
print("📄 Job Details")
print("=" * 80)

job_details = client.dataframer.transform_jobs.retrieve(job_id)

print(f"Job ID:           {job_details.id}")
print(f"Name:             {job_details.name}")
print(f"Status:           {job_details.status}")
print(f"Dataset:          {job_details.dataset_name} ({job_details.datasets_id})")
print(f"Detection method: {job_details.detection_method}")
print(f"PII types:        {job_details.pii_types}")
print(f"Threshold:        {job_details.threshold}")
print(f"Duration:         {job_details.duration_seconds}s")
print(f"Started:          {job_details.started_at}")
print(f"Completed:        {job_details.completed_at}")

### Step 7: Fetch Masked Content of Each Sample

Retrieve the anonymized content for every sample in the dataset. Each response includes the masked text and a per-entity-type count summary.

In [ ]:
print("=" * 80)
print("🔍 Masked Sample Content")
print("=" * 80)

num_samples = max(len((job_result.metrics_json or {}).get("transformed_samples", [])), 1)
entity_rows = []

def _count_entities(v):
    """Recursively sum entity counts from a flat or nested summary dict."""
    if isinstance(v, int):
        return v
    if isinstance(v, dict):
        return sum(_count_entities(x) for x in v.values())
    return 0


for idx in range(num_samples):
    sample = client.dataframer.transform_jobs.file_content(job_id, sample_index=idx)

    print(f"\nSample {idx} — {sample.file_name}")
    print(f"  File type:      {sample.file_type}")
    print(f"  Entities found: {len(sample.entities_found or [])}")
    print(f"  Entity summary: {sample.entity_summary}")
    print(f"  Masked preview:")
    print(f"  {(sample.content or '')[:400]!r}")

    if sample.entity_summary:
        for entity_type, count in sample.entity_summary.items():
            entity_rows.append({"sample": idx, "entity_type": entity_type, "count": _count_entities(count)})

### Step 8: Download All Transformed Files as ZIP

Retrieve a presigned S3 URL and download all anonymized files as a single ZIP archive. The URL is valid for 1 hour.

In [ ]:
print("=" * 80)
print("📥 Downloading ZIP archive")
print("=" * 80)

download = client.dataframer.transform_jobs.download_all(job_id)
print("Presigned URL obtained")
print(f"  Filename:   {download.filename}")
print(f"  File count: {download.file_count}")
print(f"  Size:       {download.size_bytes} bytes")

zip_response = requests.get(download.download_url)
zip_response.raise_for_status()

output_file = Path(f"transformed_{job_id[:8]}.zip")
output_file.write_bytes(zip_response.content)
print(f"\n✅ ZIP saved: {output_file.absolute()} ({output_file.stat().st_size:,} bytes)")

## Results

Entity detection summary across all samples.

In [ ]:
pd.set_option("display.max_colwidth", 80)

def _to_int(v):
    """Flatten a plain int or nested dict count to an int."""
    if isinstance(v, int):
        return v
    if isinstance(v, dict):
        return sum(_to_int(x) for x in v.values())
    return 0


if entity_rows:
    results_df = pd.DataFrame(entity_rows)
    results_df["count"] = results_df["count"].apply(_to_int)
    results_df = (
        results_df
        .groupby("entity_type")["count"]
        .sum()
        .sort_values(ascending=False)
        .reset_index()
        .rename(columns={"entity_type": "Entity Type", "count": "Total Detected"})
    )
    print("Entity detection totals across all samples:\n")
    print(results_df.to_string(index=False))
else:
    print("No entities detected — check your pii_types configuration.")

---

## What's Next?

- **Adjust `pii_types`**: add or remove entity types from the [full catalogue](#appendix-pii-types) to target exactly the entities relevant to your use case
- **Try a different `detection_method`**: switch to `heuristics` for faster runs, or `all` for maximum coverage (requires a `model_name` argument)
- **Use your own data**: replace the synthetic CSV with a real dataset via `create_with_files` or `create_from_zip`
- **Scale up**: the same workflow supports `MULTI_FILE` and `MULTI_FOLDER` datasets — pass multiple file handles or use `dataset_type="MULTI_FOLDER"` with `folder_names`
- **Integrate downstream**: the anonymized ZIP can be stored in S3, fed into further processing pipelines, or used as safe input to your LLM workflows

In [ ]:
# Uncomment to remove the resources created in this notebook
# client.dataframer.transform_jobs.delete(job_id=job_id)
# client.dataframer.seed_datasets.delete(dataset_id=dataset_id)

---

## Appendix: Available PII/PHI Types

Pass any combination of the keys below as the `pii_types` argument when creating a transform job. Each key maps to a default mask token shown in the **Masked as** column.

### Personal

| Key | Masked as |
|---|---|
| `first_name` | `<FIRST NAME>` |
| `last_name` | `<LAST NAME>` |
| `date_of_birth` | `<DOB>` |
| `date` | `<DATE>` |
| `age` | `<AGE>` |
| `gender` | `<GENDER>` |
| `nationality` | `<NATIONALITY>` |
| `race_ethnicity` | `<RACE ETHNICITY>` |
| `marital_status` | `<MARITAL STATUS>` |

### Contact

| Key | Masked as |
|---|---|
| `email` | `<EMAIL>` |
| `phone_number` | `<PHONE>` |
| `street_address` | `<ADDRESS>` |
| `postcode` | `<ZIP>` |
| `city` | `<CITY>` |
| `state` | `<STATE>` |
| `country` | `<COUNTRY>` |

### Financial

| Key | Masked as |
|---|---|
| `ssn` | `<SSN>` |
| `credit_debit_card` | `<CREDIT CARD>` |
| `bank_routing_number` | `<BANK ROUTING>` |
| `routing_number` | `<ROUTING>` |
| `tax_id` | `<TAX ID>` |
| `iban` | `<IBAN>` |

### Digital

| Key | Masked as |
|---|---|
| `ipv4` | `<IP ADDRESS>` |
| `url` | `<URL>` |
| `user_name` | `<USERNAME>` |
| `password` | `<PASSWORD>` |
| `mac_address` | `<MAC ADDRESS>` |
| `device_identifier` | `<DEVICE ID>` |

### Identity Documents

| Key | Masked as |
|---|---|
| `passport_number` | `<PASSPORT>` |
| `certificate_license_number` | `<LICENSE>` |
| `national_id` | `<NATIONAL ID>` |
| `voter_id` | `<VOTER ID>` |

### Medical / PHI

| Key | Masked as |
|---|---|
| `medical_record_number` | `<MRN>` |
| `diagnosis` | `<DIAGNOSIS>` |
| `medication` | `<MEDICATION>` |
| `health_plan_beneficiary_number` | `<HEALTH PLAN>` |
| `patient_id` | `<PATIENT ID>` |
| `lab_result` | `<LAB RESULT>` |

### Professional

| Key | Masked as |
|---|---|
| `company_name` | `<COMPANY>` |
| `occupation` | `<OCCUPATION>` |
| `employee_id` | `<EMPLOYEE ID>` |
| `salary` | `<SALARY>` |
